# `sklm`: one fine-tune, every tabular task

`sklm` exposes a fine-tuned autoregressive language model as a family of scikit-learn estimators:
a classifier, a regressor, a missing-value imputer, an imbalanced-learn oversampler and a table
synthesizer. They are five *views* of **one** fitted model. This notebook shows the shared core;
the task notebooks (`01`–`09`) put it to work.

## The one mechanism

1. Each table row is **serialized to text**, for example the JSON
   `{"sepal length": 5.1, ..., "species": "setosa"}`.
2. A small autoregressive LM is **fine-tuned** on those strings, with the **column order of each
   row permuted on every epoch**.

An autoregressive model factorizes a token sequence as

$$ \log p_\theta(t_1,\dots,t_T) = \sum_{i=1}^{T} \log p_\theta\!\left(t_i \mid t_1,\dots,t_{i-1}\right). $$

A fixed column order would teach the model one factorization. Permuting the order across epochs
trains it on many, so it learns to condition **any** column on **any subset** of the others:

$$ p\!\left(x_j \mid x_S\right)\qquad\text{for any } j \text{ and } S \subseteq \{1,\dots,d\}\setminus\{j\}. $$

Each estimator is a choice of which columns go into the prompt $x_S$ and which column $x_j$ the
model produces:

| Estimator | Conditions on (prompt) | Produces (target) | How it reads the answer |
|-----------|------------------------|-------------------|-------------------------|
| `LanguageModelClassifier` | all features | the class label | **scores** each candidate label |
| `LanguageModelRegressor` | all features | the numeric target | **generates** the value `n` times and averages, or **scores** a grid |
| `LanguageModelImputer` | a row's observed cells | that row's missing cells | **scores** categorical levels, generates or scores numbers |
| `LanguageModelOverSampler` | a minority class label | the features | **generates** synthetic rows |
| `LanguageModelSynthesizer` | nothing (or fixed columns) | whole rows | **generates** column by column |

Two reading strategies cover all of them: **scoring** a fixed candidate set (deterministic) and
**generating** a value as text.

In [1]:
from random import Random

from sklm import Field, JSONSerializer

SEED = 42

## What the model is trained on

A `Serializer` turns each row into the string the model sees; the default is JSON. One Iris row as
a list of `Field`s and its serialized form:

In [2]:
row = [
    Field(name="sepal length", value=5.1, numeric=True),
    Field(name="sepal width", value=3.5, numeric=True),
    Field(name="petal length", value=1.4, numeric=True),
    Field(name="petal width", value=0.2, numeric=True),
    Field(name="species", value="setosa", numeric=False),
]
print(JSONSerializer().serialize(row))

{"sepal length": 5.1, "sepal width": 3.5, "petal length": 1.4, "petal width": 0.2, "species": "setosa"}


## The permutation

During training the same row is emitted under many column orders. A few of the orderings the
model sees across epochs:

In [3]:
rng = Random(SEED)
for _ in range(3):
    order = row[:]
    rng.shuffle(order)
    print(JSONSerializer().serialize(order))

{"petal width": 0.2, "sepal width": 3.5, "petal length": 1.4, "species": "setosa", "sepal length": 5.1}
{"petal width": 0.2, "petal length": 1.4, "sepal length": 5.1, "species": "setosa", "sepal width": 3.5}
{"petal width": 0.2, "sepal width": 3.5, "petal length": 1.4, "sepal length": 5.1, "species": "setosa"}


## Where to go next

- `01-iris-classifier` — score the label set (classification)
- `02-autompg-regressor` — predict a number by sampling or scoring (regression)
- `03-penguins-imputer` — fill missing numeric and categorical cells (imputation)
- `04-imbalanced-oversampler` — synthesize minority-class rows (oversampling)
- `05-conditional-queries` — drive `TabularLanguageModel` directly, scoring and generating
- `06-optuna-search`, `07-stratified-cv` — tuning and evaluation with the scikit-learn contract
- `08-synthesizer` — sample whole rows from the learned joint
- `09-numeric-noise` — the `numeric_noise` training knob for numeric columns